## Import Libraries

In [12]:
import numpy as np
import cv2
import os
from scipy.spatial import distance
import matplotlib.pyplot as plt
import imageio

## Dataloder

In [13]:
class Dataloader:
    def __init__(self, data_path):
        self.root_path = data_path
    def load_img(self):
        imgs = []
        for i in range(2):
            img_path = os.path.join(self.root_path, f'image{i + 1}.png')
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            imgs.append(img)
        return imgs

## Kernel K-means

In [31]:
class KernelKMeans():
    def __init__(self, img_num, gamma_s, gamma_c, K = 2, k_means_plus_plus = False):
        self.gamma_s, self.gamma_c = gamma_s, gamma_c
        self.K = K
        self.img_num = img_num
        self.dataloader = Dataloader('./')
        self.img = self.dataloader.load_img()[img_num - 1]
        self.cluster_img = []
        self.init = k_means_plus_plus
    def kernel(self):
        x, y = np.meshgrid(range(100), range(100), indexing='ij')
        coordinates = np.column_stack((x.ravel(), y.ravel()))
        spatial_square_norm = distance.cdist(coordinates, coordinates, metric='sqeuclidean')
        flatten_img = self.img.reshape(-1, 3)
        color_square_norm = distance.cdist(flatten_img, flatten_img, metric='sqeuclidean')
        return np.exp(-self.gamma_s * spatial_square_norm) * np.exp(-self.gamma_c * color_square_norm)
    def kernel_kmeans(self, max_iter = 200):
        K = self.kernel()
        pixel_labels = np.random.randint(0, self.K, 10000)
        if self.init == True:
            pixel_labels = self.k_means_plus_plus(K)
        for _ in range(max_iter):
            pixel_membership = np.zeros((10000, self.K))
            pixel_membership[np.arange(10000), pixel_labels] = 1
            self.save_cluster(pixel_labels)
            cluster_sums = pixel_membership.T @ K @ pixel_membership     # (self.K , self.K) => sum of each cluster
            cluster_sizes = pixel_membership.sum(axis=0)[:, np.newaxis]  # (self.K, 1) => size of each cluster
            distances = np.diag(K)[:, np.newaxis] - 2 * (K @ pixel_membership / cluster_sizes.T)
            + (cluster_sums / cluster_sizes / cluster_sizes.T).sum(axis=0)
            new_labels = np.argmin(distances, axis=1)
            if np.all(pixel_labels == new_labels):
                break
            pixel_labels = new_labels
        return pixel_labels
    def save_cluster(self, labels):
        reshaped_labels = labels.reshape(100, 100, 1)
        reshaped_labels = (255 // self.K * reshaped_labels).astype(np.uint8)
        self.cluster_img.append(reshaped_labels)
    def make_gif(self):
        gif_path = f'./kmean_image{self.img_num}_{self.K}.gif'
        if self.init:
            gif_path = f'./kmean_image{self.img_num}_{self.K}_plus.gif'
        for i in range(len(self.cluster_img)):
            self.cluster_img[i] = cv2.applyColorMap(self.cluster_img[i], 2)
        imageio.mimsave(gif_path, self.cluster_img, duration=10)
        print(f'Converge in {len(self.cluster_img)} ierations')
    def k_means_plus_plus(self, K):
        centroids = []
        first_centroid = np.random.randint(0, 10000)
        centroids.append(first_centroid)
        for _ in range(1, self.K):
            distances = np.zeros(10000)
            for i in range(10000):
                cluster_sums = np.sum(K[i, centroids])
                cluster_sizes = len(centroids)
                distances[i] = (
                K[i, i]
                - 2 * cluster_sums / cluster_sizes
                + np.sum(K[np.ix_(centroids, centroids)]) / (cluster_sizes**2)
                )
            prob = distances / np.sum(distances)
            next_centroid = np.random.choice(10000, p=prob)
            centroids.append(next_centroid)
        labels = np.zeros(10000, dtype=int)
        for i in range(10000):
            distances = np.full(self.K, np.inf)
            for j, c in enumerate(centroids):
                distances[j] = K[i, i] - 2 * K[i, c] + K[c, c]
            labels[i] = np.argmin(distances)
        return labels

## Experiment

In [15]:
part_1_1 = KernelKMeans(1, 1e-3, 1e-3, K = 2)
part_1_1.kernel_kmeans()
part_1_2 = KernelKMeans(2, 1e-3, 1e-3, K = 2)
part_1_2.kernel_kmeans()

array([1, 1, 1, ..., 1, 0, 1])

In [16]:
part_1_1.make_gif()
part_1_2.make_gif()

Converge in 26 ierations
Converge in 52 ierations


In [26]:
part_2_1 = KernelKMeans(1, 1e-4, 1e-3, K = 3)
part_2_1.kernel_kmeans()
part_2_2 = KernelKMeans(2, 1e-4, 1e-3, K = 3)
part_2_2.kernel_kmeans()
part_2_3 = KernelKMeans(1, 1e-4, 1e-3, K = 4)
part_2_3.kernel_kmeans()
part_2_4 = KernelKMeans(2, 1e-4, 1e-3, K = 4)
part_2_4.kernel_kmeans()

array([0, 0, 0, ..., 3, 2, 3])

In [27]:
part_2_1.make_gif()
part_2_2.make_gif()

Converge in 34 ierations
Converge in 27 ierations


In [28]:
part_2_3.make_gif()
part_2_4.make_gif()

Converge in 39 ierations
Converge in 38 ierations


In [35]:
part_3_1 = KernelKMeans(1, 1e-4, 1e-3, K = 3, k_means_plus_plus=True)
part_3_2 = KernelKMeans(2, 1e-4, 1e-3, K = 3, k_means_plus_plus=True)
part_3_1.kernel_kmeans()
part_3_2.kernel_kmeans()

array([1, 0, 0, ..., 2, 0, 2])

In [36]:
part_3_1.make_gif()
part_3_2.make_gif()

Converge in 14 ierations
Converge in 50 ierations
